# InsightGuard AI

## Phase 4: ETL Pipeline Validation

This notebook validates the output generated by the
InsightGuard-AI ETL pipeline.

The objective is to verify:

- Clean datasets were generated successfully
- Row counts before and after transformation
- Missing values
- Key integrity
- Referential integrity
- Business rule validation

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
pd.set_option("display.max_columns", None)

In [2]:
from pathlib import Path

import pandas as pd

from IPython.display import display

pd.set_option("display.max_columns", None)

In [4]:
from pathlib import Path


def find_project_root(start_path: Path) -> Path:

    start_path = start_path.resolve()

    for path in [start_path, *start_path.parents]:

        if (
            (path / "data").exists()
            and (path / "src").exists()
        ):
            return path

    raise FileNotFoundError(
        "Project root could not be found."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

CLEAN_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "clean"
)

REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "etl"
)

print("Project Root:")
print(PROJECT_ROOT)

print("\nReports Path:")
print(REPORT_PATH)

print("\nFiles in Reports:")
print(list(REPORT_PATH.glob("*")))

Project Root:
C:\Users\Impana\OneDrive\Data Analyst Tutorial\Project\InsightGuard AI

Reports Path:
C:\Users\Impana\OneDrive\Data Analyst Tutorial\Project\InsightGuard AI\reports\etl

Files in Reports:
[WindowsPath('C:/Users/Impana/OneDrive/Data Analyst Tutorial/Project/InsightGuard AI/reports/etl/business_rule_validation.csv'), WindowsPath('C:/Users/Impana/OneDrive/Data Analyst Tutorial/Project/InsightGuard AI/reports/etl/clean_dataset_profile.csv'), WindowsPath('C:/Users/Impana/OneDrive/Data Analyst Tutorial/Project/InsightGuard AI/reports/etl/key_validation_report.csv'), WindowsPath('C:/Users/Impana/OneDrive/Data Analyst Tutorial/Project/InsightGuard AI/reports/etl/missing_value_report.csv'), WindowsPath('C:/Users/Impana/OneDrive/Data Analyst Tutorial/Project/InsightGuard AI/reports/etl/raw_dataset_profile.csv'), WindowsPath('C:/Users/Impana/OneDrive/Data Analyst Tutorial/Project/InsightGuard AI/reports/etl/referential_integrity_report.csv')]


In [5]:
raw_profile = pd.read_csv(
    REPORT_PATH / "raw_dataset_profile.csv"
)

clean_profile = pd.read_csv(
    REPORT_PATH / "clean_dataset_profile.csv"
)

comparison = (
    raw_profile
    .merge(
        clean_profile,
        on="dataset",
        suffixes=(
            "_raw",
            "_clean"
        )
    )
)

comparison["rows_removed"] = (
    comparison["rows_raw"]
    -
    comparison["rows_clean"]
)

comparison["missing_values_change"] = (
    comparison["missing_values_raw"]
    -
    comparison["missing_values_clean"]
)

display(comparison)

,dataset,rows_raw,columns_raw,missing_values_raw,exact_duplicate_rows_raw,rows_clean,columns_clean,missing_values_clean,exact_duplicate_rows_clean,rows_removed,missing_values_change
0,customers,99441,5,0,0,99441,5,0,0,0,0
1,geolocation,1000163,5,0,261831,738332,5,0,0,261831,0
2,order_items,112650,7,0,0,112650,7,0,0,0,0
3,order_payments,103886,5,0,0,103886,5,0,0,0,0
4,order_reviews,99224,7,145903,0,99224,7,145903,0,0,0
5,orders,99441,8,4908,0,99441,8,4908,0,0,0
6,products,32951,9,2448,0,32951,9,2448,0,0,0
7,sellers,3095,4,0,0,3095,4,0,0,0,0
8,category_translation,71,2,0,0,71,2,0,0,0,0


In [8]:
key_validation = pd.read_csv(
    REPORT_PATH
    / "key_validation_report.csv"
)

display(key_validation)

,dataset,key_columns,total_rows,missing_key_rows,duplicate_key_rows,is_valid
0,customers,customer_id,99441,0,0,True
1,orders,order_id,99441,0,0,True
2,products,product_id,32951,0,0,True
3,sellers,seller_id,3095,0,0,True
4,order_items,"order_id, order_item_id",112650,0,0,True
5,order_payments,"order_id, payment_sequential",103886,0,0,True


In [9]:
referential_integrity = pd.read_csv(
    REPORT_PATH
    / "referential_integrity_report.csv"
)

display(referential_integrity)

,relationship,total_child_records,unmatched_records,integrity_valid
0,orders.customer_id -> customers.customer_id,99441,0,True
1,order_items.order_id -> orders.order_id,112650,0,True
2,order_items.product_id -> products.product_id,112650,0,True
3,order_items.seller_id -> sellers.seller_id,112650,0,True
4,order_payments.order_id -> orders.order_id,103886,0,True
5,order_reviews.order_id -> orders.order_id,99224,0,True


In [10]:
business_rules = pd.read_csv(
    REPORT_PATH
    / "business_rule_validation.csv"
)

display(business_rules)

,dataset,rule,invalid_records
0,order_items,price >= 0,0
1,order_items,freight_value >= 0,0
2,order_payments,payment_value >= 0,0
3,order_payments,payment_installments >= 0,0
4,order_reviews,review_score between 1 and 5,0


In [11]:
missing_values = pd.read_csv(
    REPORT_PATH
    / "missing_value_report.csv"
)

missing_columns = missing_values[
    missing_values["missing_count"] > 0
].sort_values(
    "missing_percentage",
    ascending=False
)

display(missing_columns)

,dataset,column,missing_count,missing_percentage
25,order_reviews,review_comment_title,87656,88.3415
26,order_reviews,review_comment_message,58247,58.7025
35,orders,order_delivered_customer_date,2965,2.9817
38,products,product_category_name,610,1.8512
39,products,product_name_lenght,610,1.8512
40,products,product_description_lenght,610,1.8512
41,products,product_photos_qty,610,1.8512
34,orders,order_delivered_carrier_date,1783,1.7930
33,orders,order_approved_at,160,0.1609
42,products,product_weight_g,2,0.0061
